# Briefsummary Parser

Our high throughput dft code returns text files with a summary of calculation results. Here we convert this text files to a pandas dataframe containing all the information

We use some tools developed for this propose alone. The module `Featurizer` parses the strings in the `briefsummary.dat` files and recreates information of each sample as magnetic configuration, lattice occupation signatures, etc.

In [ ]:
from Tools.DatasetTools.Commoms import *
#sys.path.insert(0, '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.brief_summary_parser import StructSummaryParser
os.environ['PATH'] += ':'+os.path.join(os.getcwd(), 'dependencies/bopfox/src')
dataset = 'Fe-Mo' # 'Cr-Co-W',  'Fe-Mo'
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer import Featurizer

In [ ]:
!which bopfox

In [ ]:
from distutils.spawn import find_executable

In [ ]:
find_executable('bopfox')

# Load Brief Summary

In [ ]:
BS = StructSummaryParser(dataset, ForceMakeBS=True).BriefSummary

In [ ]:
!which bopfox

In [ ]:
BS.shape

In [ ]:
BS = BS[~BS.index.str.contains(r'\..*[UD]+$')]

In [ ]:
BS.shape

## Prepare targets 

One target still missing is formation Energy. Some Convenience functions to do this has been set

In [ ]:
Features  = Featurizer(BS)

In [ ]:
if 'Fe' in dataset:
    ground_states= Features.get_ground_states_energies(force_mag_phase=('Fe_pv', 'NM', 'fcc'))
else:
    ground_states = Features.get_ground_states_energies()

In [ ]:
ground_states

In [ ]:
Features.get_ground_states_energies(force_mag_phase=('Fe_pv', 'NM', 'fcc'))

In [ ]:
ground_states

As seen, at this point ground states are badly determined. They will be calculated after sanitation of dft data

In [ ]:
fig, ax =  plt.subplots()
for target_case in ['E0']:
    ax = BS[target_case].plot.hist(ax=ax, label = target_case)
ax.legend()

In [ ]:
BS

# Magnetic vs Non Magnetic

In [ ]:
target_case = 'E0'

In [ ]:
E0FM = BS[target_case][Features.Mag == 'FM']

In [ ]:
E0NM = BS[target_case][Features.Mag == 'NM']

In [ ]:
E0FM.index = E0FM.index.str.replace('.FM', '')

In [ ]:
E0NM.index = E0NM.index.str.replace('.NM', '')

In [ ]:
differences = E0NM.index.symmetric_difference(E0FM.index.str.replace('.FM', ''))

In [ ]:
differences

In [ ]:
BS[target_case][Features.StrucNames == 'fcc']

In [ ]:
BS[target_case][Features.StrucNames == 'bcc']

In [ ]:
BS[target_case][Features.StrucNames == 'hcp']

In [ ]:
DE_mag  = E0NM - E0FM 

In [ ]:
DE_mag[ abs(DE_mag > 0.1)]

In [ ]:
DE_mag[DE_mag < 0 ]

# some graphs for Raw 

In [ ]:
plt.rc('figure', figsize=(18, 8))
plt.rc('font', size=26)
plt.rc('xtick', labelsize=26)
plt.rc('ytick', labelsize=26)
plt.rc('axes', labelsize=30)
from matplotlib.lines import Line2D

In [ ]:
Features.StrucNames

In [ ]:
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.struct_db import struct_db
#struct_db = SourceFileLoader('struct_db','BopFoxFeaturizer/struct_db.py').load_module().struct_db
strucdic = struct_db().strucstrings

Target_Class = pd.Series(
    BS.index.str.split('.').map(lambda l: l[1]).map(lambda s: s.split('-')[0]),
    index=BS.index
)
Target_Class[Target_Class.map(lambda s: s in strucdic['list.hcp'])]='hcp'
Target_Class[Target_Class.map(lambda s: s in strucdic['list.fcc'])]='fcc'
Target_Class[Target_Class.map(lambda s: s in strucdic['list.bcc'])]='bcc'
Target_Class[Features.Struc == 'hcp'] = 'hcp'
Target_Class[Features.Struc == 'bcc'] = 'bcc'
Target_Class[Features.Struc == 'fcc'] = 'fcc'
Target_Class[Features.Struc.str.contains('SQS-fcc')] = 'fcc'
Target_Class[Features.Struc.str.contains('SQS-L12')] = 'fcc'
Target_Class[Features.Struc.str.contains('sigma_')] = 'sigma'

Target_Class[    
    Target_Class.str.contains('Al42W') |\
    Target_Class.str.contains('Al9Co2') |\
    Target_Class.str.contains('Al5W') |\
    Target_Class.str.contains('Al12W') |\
    Target_Class.str.contains('Al4W') |\
    Target_Class.str.contains('Al5Co2')
] = 'others'

In [ ]:
def do_mag(tag):
    if tag:
        return 'FM'
    else:
        return 'NM'

In [ ]:
BS['Mag'] = BS.index.str.contains('FM')

In [ ]:
BS['Mag'] = BS['Mag'].map(do_mag)

In [ ]:
BS['Mag']

In [ ]:
BS['Phase'] = Target_Class

In [ ]:
BS = BS[~BS.index.str.contains('delta')]

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))
sns.histplot(y=BS['Phase'], ax=ax, hue = BS['Mag'], multiple='stack', palette=['mediumseagreen', 'darkblue'], binwidth=0.1)
stack_histogram_file_name = os.path.join(dataset, 'graphs', f'{dataset}_StackCounts_RAW.pdf')
fig.tight_layout()
if not os.path.exists(os.path.dirname(stack_histogram_file_name)):
    os.makedirs(os.path.dirname(stack_histogram_file_name))
fig.savefig(stack_histogram_file_name)

In [ ]:
BS.to_pickle(f'{dataset}/ParsedBriefsummary.pkl')